In [1]:
from PIL import Image
from transformers import AutoProcessor, AutoModel
import torch

/data/geraugi/anaconda3/envs/biovil-t-env/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
checkpoint = "microsoft/BiomedVLP-BioViL-T"
processor = AutoProcessor.from_pretrained(checkpoint, trust_remote_code=True)
model     = AutoModel.from_pretrained(checkpoint, trust_remote_code=True)
model.eval()

Some weights of the model checkpoint at microsoft/BiomedVLP-BioViL-T were not used when initializing CXRBertModel: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight']
- This IS expected if you are initializing CXRBertModel from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing CXRBertModel from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


CXRBertModel(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.25, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSdpaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.25, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e-12, elementwis

In [3]:
import pandas as pd

In [ ]:
output_dir = '/data/geraugi/chexfound/dataset/training/'
testset_file = f'{output_dir}/test.csv'
testset_df = pd.read_csv(testset_file)
testset_df = testset_df.set_index("dicom_id", drop=True)
# filter by dicom_id tar_path, member_path, offset, and size columns
testset_df = testset_df[['tar_path', 'member_path', 'offset', 'size']]
testset_df.head(3)

In [ ]:
plural_dataset = f'/data/geraugi/plural/pre_processed_data/compact_context_w_ref_id.csv'
plural_df = pd.read_csv(plural_dataset)
# filter by split == test
plural_df = plural_df[plural_df['split'] == 'test']
# filter by dicom_id and ref_dicom_id columns
dicom2ref_dicom_df = plural_df[['dicom_id', 'ref_dicom_id']]
# drop dicom_id duplicates
dicom2ref_dicom_df = dicom2ref_dicom_df.drop_duplicates(subset=['dicom_id'])
dicom2ref_dicom_df.head(3)


In [19]:
cols = ["tar_path", "member_path", "offset", "size"]
testset_df = testset_df.rename_axis("dicom_id")
base = dicom2ref_dicom_df.set_index("dicom_id", verify_integrity=True).reset_index()

merged = (base
    .merge(testset_df[cols], left_on="dicom_id", right_index=True, how="left")
    .merge(testset_df[cols].add_prefix("ref_"),
           left_on="ref_dicom_id", right_index=True, how="left")
    .set_index("dicom_id")
)

In [ ]:
merged

In [ ]:
import io
def get_image_data( dataset, index: int):
        #img_path = os.path.join(self.base_dir, self.labels.iloc[index]['fpath'])
        row = dataset.iloc[index]
        tar_path = row["tar_path"]
        offset   = int(row["offset"])
        size     = int(row["size"])
        ref_tar_path = row["ref_tar_path"]
        ref_offset   = int(row["ref_offset"])
        ref_size     = int(row["ref_size"])
        
        # 1) Lese nur den relevanten Bereich aus dem TAR
        with open(tar_path, "rb") as f:
            f.seek(offset)
            raw = f.read(size)
        with open(ref_tar_path, "rb") as ref_f:
            ref_f.seek(ref_offset)
            ref_raw = ref_f.read(ref_size)
        # In-Painting: lade Image aus Bytes
        img = Image.open(io.BytesIO(raw))
        img.convert(mode="RGB")
        # display image
        img.show()
        ref_img = Image.open(io.BytesIO(ref_raw))
        ref_img.convert(mode="RGB")
        # display image
        ref_img.show()
        return [img, ref_img]
i=0
images = get_image_data(testset_df, i)
#j = 0
#for i in range(5):
#    j+=0
#    img = get_image_data(testset_df, i+j)
#    print(f"Image {i} loaded successfully.")


In [25]:

import io
from pathlib import Path
from typing import Tuple

import torch
import torch.nn.functional as F
from PIL import Image
from transformers import AutoTokenizer, AutoModel

# --- HI-ML / BioViL-T image side (multi-image encoder) ---
from health_multimodal.image.model.pretrained import get_biovil_t_image_encoder
from health_multimodal.image.data.transforms import create_chest_xray_transform_for_inference

In [30]:
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ------------------------
# DataFrame -> load two CXRs from TAR slices
# ------------------------
def load_pair_from_df(df, index: int) -> Tuple[Image.Image, Image.Image]:
    """
    Expects columns:
      tar_path, offset, size,
      ref_tar_path, ref_offset, ref_size
    Returns (curr_img, ref_img) as PIL RGB.
    """
    row = df.iloc[index]

    def _read_one(path_col, off_col, size_col):
        tar_path = row[path_col]
        offset   = int(row[off_col])
        size     = int(row[size_col])
        if not isinstance(tar_path, str):
            raise ValueError(f"Missing {path_col} at row {index}")
        with open(tar_path, "rb") as f:
            f.seek(offset)
            raw = f.read(size)
        img = Image.open(io.BytesIO(raw)).convert("L")
        img.load()
        return img

    curr_img = _read_one("tar_path", "offset", "size")
    ref_img  = _read_one("ref_tar_path", "ref_offset", "ref_size")
    return curr_img, ref_img

# ------------------------
# Models
# ------------------------

# Text: load from HF with safetensors to avoid torch.load gating
HF_ID = "microsoft/BiomedVLP-BioViL-T"
tok = AutoTokenizer.from_pretrained(HF_ID, trust_remote_code=True)
txt_model = AutoModel.from_pretrained(
    HF_ID, trust_remote_code=True, use_safetensors=True
).to(DEVICE).eval()

# Image: BioViL-T multi-image encoder (current & previous)
img_model = get_biovil_t_image_encoder().to(DEVICE).eval()
transform = create_chest_xray_transform_for_inference(resize=512, center_crop_size=448)

def _prep(img: Image.Image) -> torch.Tensor:
    return transform(img).unsqueeze(0).to(DEVICE)  # [1,C,H,W]

# ------------------------
# Similarity: prompt vs (current, previous) pair
# ------------------------
@torch.no_grad()
def prompt_pair_similarity_from_df(df, index: int, prompt: str) -> float:
    curr_img, prev_img = load_pair_from_df(df, index)
    x_curr, x_prev = _prep(curr_img), _prep(prev_img)

    # Pair embedding via BioViL-T multi-image forward
    patch_x, pooled_x = img_model.encoder(
        current_image=x_curr,
        previous_image=x_prev,
        return_patch_embeddings=True,
    )
    out = img_model.forward_post_encoder(patch_x, pooled_x)
    e_pair = F.normalize(out.projected_global_embedding, dim=-1)   # [1, D]

    # Text embedding (BioViL-T BERT)
    toks = tok(prompt, return_tensors="pt", padding=True, truncation=True).to(DEVICE)
    te = txt_model.get_projected_text_embeddings(
        input_ids=toks.input_ids, attention_mask=toks.attention_mask
    )
    te = F.normalize(te, dim=-1)                                   # [1, D]

    # Cosine similarity of unit vectors
    return float((te * e_pair).sum())

# Example:
i = 0
score = prompt_pair_similarity_from_df(merged, i, "increased pleural effusion")
print(f"Similarity: {score:.4f}")

Some weights of the model checkpoint at microsoft/BiomedVLP-BioViL-T were not used when initializing CXRBertModel: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight']
- This IS expected if you are initializing CXRBertModel from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing CXRBertModel from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


Using downloaded and verified file: /tmp/biovil_t_image_model_proj_size_128.pt


Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.


Similarity: -0.6092


In [ ]:
gt_diff_df = pd.read_csv('/data/geraugi/plural/dataset_files/xp_out/image_difference_all_dataset_gt.csv')
gt_diff_df.head(3)

In [39]:
def build_dataset(plural_df, gt_diff_df, split='test'):
    input_dir = '/data/geraugi/chexfound/dataset/training/'
    output_dir = '/data/geraugi/biovil-t/'
    chexfound_file = f'{input_dir}/{split}.csv'
    img_path_df = pd.read_csv(chexfound_file)
    img_path_df = img_path_df.set_index("dicom_id", drop=True)
    # filter by dicom_id tar_path, member_path, offset, and size columns
    img_path_df = img_path_df[['tar_path', 'member_path', 'offset', 'size']]

    # filter by split == test
    plural_df = plural_df[plural_df['split'] == split]
    # filter by dicom_id and ref_dicom_id columns
    dicom2ref_dicom_df = plural_df[['dicom_id', 'ref_dicom_id']]
    # drop dicom_id duplicates
    dicom2ref_dicom_df = dicom2ref_dicom_df.drop_duplicates(subset=['dicom_id'])

    cols = ["tar_path", "member_path", "offset", "size"]
    img_path_df = img_path_df.rename_axis("dicom_id")
    base = dicom2ref_dicom_df.set_index("dicom_id", verify_integrity=True).reset_index()

    merged = (base
        .merge(img_path_df[cols], left_on="dicom_id", right_index=True, how="left")
        .merge(img_path_df[cols].add_prefix("ref_"),
               left_on="ref_dicom_id", right_index=True, how="left")
        .set_index("dicom_id")
    )
    # filter gt_diff_df by split==split
    gt_diff_df = gt_diff_df[gt_diff_df['split'] == split]
    # keep only columns: dicom_id, ref_dicom_id, <disease>_diff
    gt_diff_df = gt_diff_df[['dicom_id', 'ref_dicom_id'] + [col for col in gt_diff_df.columns if col.endswith('_diff')]]
    # add a column called additional_<disease_lower_case_no_spaces> with a value of 1 where <disease>_diff is 1
    # add a column called missing_<disease_lower_case_no_spaces> with a value of 1 where <disease>_diff is -1
    for col in gt_diff_df.columns:
        if col.endswith('_diff'):
            disease_lower_case_no_spaces = col[:-5].lower().replace(' ', '_')
            gt_diff_df[f'additional_{disease_lower_case_no_spaces}'] = (gt_diff_df[col] == 1).astype(int)
            # add a column called missing_<disease_lower_case_no_spaces> with a value of 1 where <disease>_diff is -1
            gt_diff_df[f'missing_{disease_lower_case_no_spaces}'] = (gt_diff_df[col] == -1).astype(int)
    # drop the original <disease>_diff columns
    gt_diff_df = gt_diff_df.drop(columns=[col for col in gt_diff_df.columns if col.endswith('_diff')])
    # build final dataset by merging meged and gt_diff_df on dicom_id and ref_dicom_id
    final_df = merged.merge(gt_diff_df, on=['dicom_id', 'ref_dicom_id'], how='left')
    # save final_df to csv
    output_file = f'{output_dir}/{split}_biovil_t_dataset.csv'
    final_df.to_csv(output_file, index=True)
    #print length of final_df
    print(f"Length of {split} df: {len(final_df)}")
    print(f"Dataset saved to {output_file}")
    return final_df

plural_dataset = f'/data/geraugi/plural/pre_processed_data/compact_context_w_ref_id.csv'
plural_df = pd.read_csv(plural_dataset)
splits = ['train', 'val', 'test']
datasets = {}
for split in splits:
    print(f"Building {split} dataset...")
    datasets[split] = build_dataset(plural_df, gt_diff_df, split=split)


Building train dataset...
Length of train df: 131823
Dataset saved to /data/geraugi/biovil-t//train_biovil_t_dataset.csv
Building val dataset...
Length of val df: 16400
Dataset saved to /data/geraugi/biovil-t//val_biovil_t_dataset.csv
Building test dataset...
Length of test df: 16431
Dataset saved to /data/geraugi/biovil-t//test_biovil_t_dataset.csv


In [ ]:
testset = datasets['test']
testset.head(3)

In [41]:
# print all columns of testset
print(testset.columns.tolist())


['dicom_id', 'ref_dicom_id', 'tar_path', 'member_path', 'offset', 'size', 'ref_tar_path', 'ref_member_path', 'ref_offset', 'ref_size', 'additional_atelectasis', 'missing_atelectasis', 'additional_pleural_effusion', 'missing_pleural_effusion', 'additional_lung_opacity', 'missing_lung_opacity', 'additional_edema', 'missing_edema', 'additional_cardiomegaly', 'missing_cardiomegaly', 'additional_pneumonia', 'missing_pneumonia', 'additional_consolidation', 'missing_consolidation', 'additional_pneumothorax', 'missing_pneumothorax']
